In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pickle

In [14]:
# Projection to a common embedding dimension
class Projection(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.smiles_proj = nn.Linear(384, 256)
        self.graph_proj = nn.Linear(128, 256)
        self.protein_proj = nn.Linear(1024, 256)

    def forward(self, e_smiles, e_graph, e_protein):
        e_smiles = self.smiles_proj(e_smiles)
        e_graph = self.graph_proj(e_graph)
        e_protein = self.protein_proj(e_protein)
        
        return e_smiles, e_graph, e_protein

In [15]:
#cross-modal fusion module
class CrossModalFusion(nn.Module):
    def __init__(self, embed_dim=256, num_heads=4):
        super().__init__()

        # Learnable fusion token
        self.fusion_token = nn.Parameter(torch.randn(1, 1, embed_dim))

        # Cross-attention: fusion token queries modalities
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.layer_norm1 = nn.LayerNorm(embed_dim)
        self.layer_norm2 = nn.LayerNorm(embed_dim)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim)
        )

    def forward(self, e_smiles, e_graph, e_protein):
        """
        Inputs:
            e_smiles  : (B, 256)
            e_graph   : (B, 256)
            e_protein : (B, 256)
        """

        B = e_smiles.size(0)

        # Stack modalities → (B, 3, 256)
        modalities = torch.stack([e_smiles, e_graph, e_protein], dim=1)

        # Expand fusion token for batch → (B, 1, 256)
        fusion_token = self.fusion_token.expand(B, -1, -1)

        # Cross-attention:
        # Query = fusion token
        # Key/Value = modalities
        attn_output, _ = self.cross_attn(
            query=fusion_token,
            key=modalities,
            value=modalities
        )

        # Residual + norm
        x = self.layer_norm1(fusion_token + attn_output)

        # Feed-forward
        ffn_output = self.ffn(x)
        x = self.layer_norm2(x + ffn_output)

        # Final fused embedding → (B, 256)
        e_fused = x.squeeze(1)

        return e_fused

In [16]:
# full pipeline
class DrugFusionModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.projection = Projection()
        self.fusion = CrossModalFusion(embed_dim=256)

    def forward(self, e_smiles, e_graph, e_protein):
        
        # Step 1: Dimension alignment
        e_smiles, e_graph, e_protein = self.projection(
            e_smiles, e_graph, e_protein
        )

        # Step 2: Cross-modal fusion
        e_fused = self.fusion(
            e_smiles, e_graph, e_protein
        )

        return e_fused

In [17]:
# Load and process
# Load
with open("drug_graph_embeddings.pkl", "rb") as f:
    graph_dict = pickle.load(f)

with open("drug_protein_embeddings.pkl", "rb") as f:
    protein_dict = pickle.load(f)

smiles_dict = torch.load("chemberta_smiles_embeddings.pt")

# Initialize model
model = DrugFusionModel()
model.eval()

fused_dict = {}

for cid in graph_dict.keys():
    
    if cid not in smiles_dict or cid not in protein_dict:
        continue
    
    e_graph = torch.tensor(graph_dict[cid]).float().unsqueeze(0)
    e_smiles = torch.tensor(smiles_dict[cid]).float().unsqueeze(0)
    e_protein = torch.tensor(protein_dict[cid]).float().unsqueeze(0)

    with torch.no_grad():
        e_fused = model(e_smiles, e_graph, e_protein)

    fused_dict[cid] = e_fused.squeeze(0)

C:\Users\Diya Budlakoti\AppData\Local\Temp\ipykernel_14860\3114544236.py:23: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  e_smiles = torch.tensor(smiles_dict[cid]).float().unsqueeze(0)


In [18]:
torch.save(fused_dict, "drug_fused_embeddings.pt")